# Daily Report

One day's export, for either issue:

- **Politics**: a treemap of the day's top keywords, where the local LLM writes each tile's topic
  and headline, and a pie of author sentiment (one vote per author). `daily_overview()`.
- **Environment**: the environment map with issue and island bars. `env_daily()`.

### **Mounting Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

### **Inputing the Requirement**

In [ ]:
import datetime as dt, os

TIMESTAMP       = dt.datetime.now().strftime('%d%m%Y')
ISSUES          = 'Politics' # 'Politics' or 'Environment'
ROBJ            = 'President Prabowo' # Changes this based on your research object
ROBJ_TERMS      = ['prabowo', 'subianto'] # Politics: words naming ROBJ, left out of the treemap keywords
PATH            = '/content/drive/MyDrive/Data/Raw Data DPR 2026' # Path to access the data in google drive
FILE_NAME       = 'dpr26_harian.xlsx' # the day's export
OUT_DIR         = f'/content/drive/MyDrive/model_result/daily_{ISSUES.lower()}_{TIMESTAMP}';os.makedirs(OUT_DIR,exist_ok=True) # To upload the result directly
N_TILES         = 20   # Politics: tiles in the treemap
N_DOCS          = 4    # Politics: representative posts per tile given to the LLM
MODEL_REPO      = 'faizonly5953/gemma2-9b-cpt-sahabatai-v1-instruct-GGUF' # Change the model based on the provider that you use
MODEL_FILE      = 'gemma2-9b-cpt-sahabatai-v1-instruct.q4_k_m.gguf' # Change the model based on the provider that you use

### **Notebook Set Up**

Politics also installs llama-cpp for the local LLM. A GPU whose driver supports CUDA 12 or newer
gets the CUDA build; anything else (CPU, or an older GPU) gets the CPU build, which is slower
but gives the same labels.

In [ ]:
%pip install -q "great[all] @ git+https://github.com/azmkto/GREAT-Tools.git"

HAS_GPU = False
if ISSUES == 'Politics':
    import re, shutil, subprocess

    smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout if shutil.which('nvidia-smi') else ''
    cuda = re.search(r'CUDA Version: (\d+)\.(\d+)', smi)
    major, minor = (int(cuda[1]), int(cuda[2])) if cuda else (0, 0)
    HAS_GPU = major >= 12
    CU_TAG = f'cu12{min(minor, 4) if major == 12 else 4}' if HAS_GPU else 'cpu'   # abetlen publishes cu121..cu124 + cpu
    print(f'Detected: CUDA {major}.{minor} -> {CU_TAG}' if cuda else f'Detected: no GPU -> {CU_TAG}')
    !pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/{CU_TAG}

In [ ]:
import json

import pandas as pd

from great import classify_issue, resolve_frame
from great.viz import checks, prep
from great.viz.environment import env_daily
from great.viz.overview import author_sentiment, daily_overview, keyword_tiles, representative_posts

### **Accessing File**

In [ ]:
df = pd.read_excel(f'{PATH}/{FILE_NAME}', header=1)
df1, start_date, end_date = prep.prepare_data(df)

print(f'{start_date} - {end_date}')

### **Checking Dataframe Info**

In [ ]:
checks.frame_info(df1, 'Main Dataframe')
checks.composition(df1)

### **Topic and Headline from the LLM**

Politics only. `keyword_tiles()` files each post under its most distinctive keyword and keeps the
`N_TILES` biggest. For each tile the LLM reads the keyword and its `N_DOCS` most repeated
headlines (`representative_posts()`), and writes the tile's topic and headline. If a call
fails, the tile keeps the keyword as its topic and its top headline, cut to 15 words.

In [ ]:
if ISSUES == 'Politics':
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama

    model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
    llm = Llama(model_path=model_path, n_gpu_layers=-1 if HAS_GPU else 0, n_ctx=4096, n_batch=512, verbose=False)

In [ ]:
FIELDS = ['topic', 'headline']
SCHEMA = {'type': 'object', 'properties': {f: {'type': 'string'} for f in FIELDS}, 'required': FIELDS}

PROMPT = """You are a media analyst at a socio-political think tank, summarising today's public discourse.

These posts all use the keyword "{keyword}":
{documents}

Return JSON with exactly these fields:
1. topic    - the topic in at most 3 words: the person, institution, or issue the posts are about.
2. headline - one news-style headline of at most 15 words saying what happened, grounded in the posts.

Answer in Indonesian. Output only JSON.
"""

if ISSUES == 'Politics':
    df1['Keyword'] = keyword_tiles(df1, exclude=ROBJ_TERMS, n_tiles=N_TILES)

    records = []
    for keyword, count in df1['Keyword'].value_counts().items():
        docs = representative_posts(df1[df1['Keyword'] == keyword], n=N_DOCS)
        try:
            r = llm.create_chat_completion(
                messages=[{'role': 'user', 'content': PROMPT.format(keyword=keyword, documents='\n'.join(f'- {d}' for d in docs))}],
                response_format={'type': 'json_object', 'schema': SCHEMA},
                temperature=0.2, max_tokens=200,
            )
            out = json.loads(r['choices'][0]['message']['content'])
        except Exception as e:
            print(f'{keyword} failed: {type(e).__name__}: {e}')
            out = {}
        records.append({
            'Keyword': keyword,
            'Count': count,
            'Topic': (out.get('topic') or keyword).strip(),
            'Headline': (out.get('headline') or ' '.join(docs[0].split()[:15])).strip(),
        })

    tiles = pd.DataFrame(records)
    tiles.to_excel(f'{OUT_DIR}/daily_politics_topics_{TIMESTAMP}.xlsx', index=False)
    display(tiles)

### **Visualize**

In [ ]:
if ISSUES == 'Politics':
    fig = daily_overview(tiles, author_sentiment(df1), ROBJ, start_date, end_date)
else:
    df1['Isu_Inti'] = df1['Mentions'].map(classify_issue)
    df1[['Provinsi', 'Pulau']] = resolve_frame(df1)
    fig = env_daily(df1, start_date=start_date, end_date=end_date)

fig.savefig(f'{OUT_DIR}/daily_{ISSUES.lower()}_{TIMESTAMP}.png', dpi=150, bbox_inches='tight')